### Final Project

### CCDB June-2026 Report

### Set Up

In [1]:
# --- Setup ---------------------------------------------------------------
# Core libraries only; model/API imports get appended once we pick the GenAI tool.
import hashlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

# Notebook lives in notebooks/; resolve paths from the project root so it runs on either partner's machine
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "raw" / "CCDB_Export_19_June_2026.zip"

# Reference fingerprint of the June 2026 export, recorded when we first profiled it (2026-10-24).
# The load step recomputes the hash from the file and compares it to this value.
EXPECTED_SHA256 = "3021c8af9eeed4971a92f15be21c7ce08cf3c1337151de255e5b12bf782049df"

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "| numpy:", np.__version__)
assert DATA_PATH.exists(), f"Data file not found: {DATA_PATH}"

Python: 3.12.10
pandas: 3.0.5 | numpy: 2.5.3


### Load Data

In [2]:
# --- Load data -----------------------------------------------------------
# Integrity check: recompute the file's fingerprint and compare it to the recorded one
actual_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
sha_match = actual_sha256 == EXPECTED_SHA256

print(f"File:            {DATA_PATH.name}")
print(f"Expected sha256: {EXPECTED_SHA256}")
print(f"Actual sha256:   {actual_sha256}")
print(f"Integrity check: {'PASS' if sha_match else 'FAIL'}")
assert sha_match, "Data file differs from the profiled export; do not continue"

# Read every column as text; types get converted on purpose in the cleaning step
df_raw = pd.read_csv(DATA_PATH, dtype=str, low_memory=False)

EXPECTED_COLS = [
    "Date received", "Product", "Sub-product", "Issue", "Sub-issue",
    "Consumer complaint narrative", "Company public response", "Company",
    "State", "ZIP code", "Tags", "Submitted via", "Date sent to company",
    "Company response to consumer", "Timely response?", "Complaint ID",
]
assert list(df_raw.columns) == EXPECTED_COLS, "Column layout differs from the profiled export"
assert len(df_raw) == 620_794, f"Unexpected row count: {len(df_raw):,}"
assert df_raw["Complaint ID"].is_unique, "Duplicate Complaint IDs"

print(f"\nLoaded {len(df_raw):,} rows x {df_raw.shape[1]} columns "
      f"({df_raw.memory_usage(deep=True).sum() / 1e6:,.0f} MB in memory)")

# Preview without the narrative column to keep the output short
df_raw.drop(columns=["Consumer complaint narrative"]).head(3)

File:            CCDB_Export_19_June_2026.zip
Expected sha256: 3021c8af9eeed4971a92f15be21c7ce08cf3c1337151de255e5b12bf782049df
Actual sha256:   3021c8af9eeed4971a92f15be21c7ce08cf3c1337151de255e5b12bf782049df
Integrity check: PASS

Loaded 620,794 rows x 16 columns (670 MB in memory)


,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2026-06-07,Credit card,General-purpose credit card or charge card,Closing your account,Company closed your account,NaN,OneMain Finance Corporation,TN,37087,NaN,Web,2026-06-07,Closed with explanation,Yes,22972138
1,2026-06-06,Credit reporting or other personal consumer reports,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't recognize,NaN,"EQUIFAX, INC.",MI,48221,NaN,Web,2026-06-06,Closed with explanation,Yes,22960234
2,2026-06-02,Debt collection,Other debt,Electronic communications,"You told them to stop contacting you, but they keep trying",NaN,"Landmark Strategy Group, LLC",NC,28278,NaN,Web,2026-08-28,Untimely response,No,22819786


### Profile: Structure

Dimensions, data types, missingness and uniqueness for every column, computed directly from the raw export. Everything here is read off the data; no outside assumptions.

In [3]:
# --- Profile: structure ----------------------------------------------------
n_rows, n_cols = df_raw.shape
print(f"Dimensions: {n_rows:,} rows x {n_cols} columns")

profile = pd.DataFrame({
    "dtype":     df_raw.dtypes.astype(str),
    "non_null":  df_raw.notna().sum(),
    "null_n":    df_raw.isna().sum(),
    "null_%":    (100 * df_raw.isna().mean()).round(1),
    "n_unique":  df_raw.nunique(),
    "unique_%":  (100 * df_raw.nunique() / n_rows).round(2),
})

# Sanity check: every row is either filled or missing for each column
assert (profile["non_null"] + profile["null_n"] == n_rows).all()

display(profile)

Dimensions: 620,794 rows x 16 columns


,dtype,non_null,null_n,null_%,n_unique,unique_%
Date received,str,620794,0,0.0,30,0.00
Product,str,620794,0,0.0,11,0.00
Sub-product,str,620794,0,0.0,55,0.01
Issue,str,620794,0,0.0,85,0.01
Sub-issue,str,613772,7022,1.1,208,0.03
Consumer complaint narrative,str,16820,603974,97.3,15380,2.48
Company public response,str,389489,231305,37.3,9,0.00
Company,str,620794,0,0.0,1831,0.29
State,str,620223,571,0.1,60,0.01
ZIP code,str,620329,465,0.1,14756,2.38


### Explore

Only complaints with a consumer narrative can be read by a language model, so first check how that subset compares with the full export.

In [5]:
# --- Explore: narrative subset vs. all complaints --------------------------
NARRATIVE_COL = "Consumer complaint narrative"
has_narrative = df_raw[NARRATIVE_COL].fillna("").str.strip() != ""
df_narrative = df_raw[has_narrative]
print(f"Complaints with a narrative: {has_narrative.sum():,} of {len(df_raw):,} ({has_narrative.mean():.1%})")

# Product mix, issue variety and text length, side by side
by_product = pd.DataFrame({
    "all_n": df_raw["Product"].value_counts(),
    "narrative_n": df_narrative["Product"].value_counts(),
    "narrative_issues": df_narrative.groupby("Product")["Issue"].nunique(),
    "narrative_median_chars": df_narrative.groupby("Product")[NARRATIVE_COL].apply(lambda s: s.str.len().median()),
}).fillna(0).astype(int)
by_product.insert(1, "all_%", (100 * by_product["all_n"] / by_product["all_n"].sum()).round(1))
by_product.insert(3, "narrative_%", (100 * by_product["narrative_n"] / by_product["narrative_n"].sum()).round(1))
by_product = by_product.sort_values("narrative_n", ascending=False)
display(by_product)

# Outcome mix within each product (narrative subset only), row percentages
outcome_by_product = pd.crosstab(df_narrative["Product"], df_narrative["Company response to consumer"],
                                 normalize="index").mul(100).round(1)
display(outcome_by_product.reindex(by_product.index))

Complaints with a narrative: 16,820 of 620,794 (2.7%)


,all_n,all_%,narrative_n,narrative_%,narrative_issues,narrative_median_chars
Product,,,,,,
Debt collection,27629,4.5,5098,30.3,7,818
Checking or savings account,7671,1.2,3270,19.4,10,1170
Credit card,7453,1.2,2840,16.9,15,1277
"Money transfer, virtual currency, or money service",3037,0.5,1340,8.0,15,1122
Mortgage,2967,0.5,1151,6.8,7,1470
Vehicle loan or lease,2002,0.3,793,4.7,9,1324
Credit reporting or other personal consumer reports,565831,91.1,751,4.5,8,674
"Payday loan, title loan, personal loan, or advance loan",1588,0.3,636,3.8,28,1131
Student loan,1724,0.3,576,3.4,9,1286


Company response to consumer,Closed with explanation,Closed with monetary relief,Closed with non-monetary relief,Untimely response
Product,,,,
Debt collection,72.5,0.3,24.3,2.9
Checking or savings account,79.4,14.2,6.2,0.2
Credit card,72.3,17.2,8.9,1.6
"Money transfer, virtual currency, or money service",82.3,9.9,7.5,0.4
Mortgage,88.9,2.2,4.6,4.3
Vehicle loan or lease,89.3,2.3,7.3,1.1
Credit reporting or other personal consumer reports,58.9,0.0,40.9,0.3
"Payday loan, title loan, personal loan, or advance loan",93.4,2.5,2.7,1.4
Student loan,60.8,0.2,1.0,38.0
